In [1]:
import sys
import torch

print("Python path:", sys.executable)
print("Torch version:", torch.__version__)

Python path: /mnt/d/git/betterbot/.venv/bin/python
Torch version: 2.14.0+cu130


In [2]:
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Capability:", torch.cuda.get_device_capability())

GPU: NVIDIA GeForce RTX 5060 Ti
PyTorch: 2.14.0+cu130
CUDA: 13.0
CUDA available: True
Capability: (12, 0)


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm_polygraph.utils.model import WhiteboxModel

model_path = "Qwen/Qwen2.5-0.5B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda:0", attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = WhiteboxModel(base_model, tokenizer, model_path=model_path)

/mnt/d/git/betterbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/d/git/betterbot/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 215.06it/s]


In [4]:
from lm_polygraph.estimators import *

ue_method = MeanTokenEntropy()

In [5]:
from lm_polygraph.utils import estimate_uncertainty

input_text = "Who is George Bush?"
ue = estimate_uncertainty(model, ue_method, input_text=input_text)
print(ue)
# UncertaintyOutput(uncertainty=-6.504108926902215, input_text='Who is George Bush?', generation_text=' President of the United States', model_path='Qwen/Qwen2.5-0.5B-Instruct')

UncertaintyOutput(uncertainty=1.4390383958816528, input_text='Who is George Bush?', generation_text=' George Bush is a former US President who served from 1989 to 2013. He was the 4th President of the United States and the 41st President of the United Kingdom. He was the first President to be born in the 20th century and the first to be born in the United States. He was also the first President to be born in the United Kingdom. Bush was a member of the Republican Party and served as the 41', generation_tokens=[9857, 14079, 374, 264, 4741, 2274, 4795, 879, 10223, 504, 220, 16, 24, 23, 24, 311, 220, 17, 15, 16, 18, 13, 1260, 572, 279, 220, 19, 339, 4795, 315, 279, 3639, 4180, 323, 279, 220, 19, 16, 267, 4795, 315, 279, 3639, 15072, 13, 1260, 572, 279, 1156, 4795, 311, 387, 9223, 304, 279, 220, 17, 15, 339, 9294, 323, 279, 1156, 311, 387, 9223, 304, 279, 3639, 4180, 13, 1260, 572, 1083, 279, 1156, 4795, 311, 387, 9223, 304, 279, 3639, 15072, 13, 14079, 572, 264, 4462, 315, 279, 9356, 8554

In [6]:
import spacy

# Load English tokenizer, tagger, parser and NER
nlp = spacy.load("en_core_web_sm")

# Process whole documents
text = ("When Sebastian Thrun started working on self-driving cars at "
        "Google in 2007, few people outside of the company took him "
        "seriously. “I can tell you very senior CEOs of major American "
        "car companies would shake my hand and turn away because I wasn’t "
        "worth talking to,” said Thrun, in an interview with Recode earlier "
        "this week.")
doc = nlp(text)

# Analyze syntax
print("Noun phrases:", [chunk.text for chunk in doc.noun_chunks])
print("Verbs:", [token.lemma_ for token in doc if token.pos_ == "VERB"])

# Find named entities, phrases and concepts
for entity in doc.ents:
    print(entity.text, entity.label_)

Noun phrases: ['Sebastian Thrun', 'self-driving cars', 'Google', 'few people', 'the company', 'him', 'I', 'you', 'very senior CEOs', 'major American car companies', 'my hand', 'I', 'Thrun', 'an interview', 'Recode']
Verbs: ['start', 'work', 'drive', 'take', 'tell', 'shake', 'turn', 'talk', 'say']
Sebastian Thrun PERSON
Google ORG
2007 DATE
American NORP
Thrun GPE
Recode ORG
earlier this week DATE


In [12]:
import pickle
import numpy as np # נוודא ש-numpy נטען

idf_file_path = "focus/token_frequency_data/token_idf_RedPajama-INCITE-7B-Base.pkl" 

with open(idf_file_path, 'rb') as f:
    token_idf = pickle.load(f)

# במקום len, נבדוק את הצורה (shape) של המערך
print(f"Loaded IDF array with shape: {token_idf.shape}")

# נדפיס את 10 הערכים הראשונים (טוקנים 0 עד 9)
print("First 10 IDF values:", token_idf[:10])

Loaded IDF array with shape: (50277,)
First 10 IDF values: [13.05034522 13.7434924   1.35449421  1.54335735  3.4186992   3.55354853
  2.16445153  2.84894065  1.77305981  1.99792464]


In [14]:
from transformers import AutoTokenizer

# טעינת הטוקנייזר של RedPajama (הטוקנייזר בלבד, לא המודל הכבד)
rp_tokenizer = AutoTokenizer.from_pretrained("togethercomputer/RedPajama-INCITE-7B-Base")

def get_word_idf(word: str, idf_array: np.ndarray, tokenizer) -> float:
    # המרת המילה (מחרוזת) ל-ID ספציפי של RedPajama
    token_ids = tokenizer.encode(word, add_special_tokens=False)
    
    if not token_ids:
        return 1.0 # ערך ברירת מחדל אם המילה לא נמצאה
        
    # אם המילה חולקה לכמה תת-מילים, ניקח את ה-IDF של הראשונה (או שנעשה ממוצע)
    main_token_id = token_ids[0]
    
    # שליפת הציון מהמערך שהעלינו לזיכרון
    return idf_array[main_token_id]

# בוא נבדוק שהגשר עובד!
words_to_test = ["the", "1992", "France", "George"]

for w in words_to_test:
    score = get_word_idf(w, token_idf, rp_tokenizer)
    print(f"IDF of '{w}': {score:.4f}")

IDF of 'the': 1.8811
IDF of '1992': 5.3729
IDF of 'France': 5.4269
IDF of 'George': 4.8472


In [15]:
def analyze_focus_hallucination(text: str, idf_array, tokenizer, nlp_model):
    # 1. ניתוח הטקסט בעזרת Spacy
    doc = nlp_model(text)
    
    # 2. סינון ישויות בלבד (אנשים, תאריכים, מקומות וכו')
    entities = [ent for ent in doc.ents]
    
    if not entities:
        print("No entities found for FOCUS analysis.")
        return
        
    print(f"Analyzing Text: '{text}'\n")
    print(f"{'Entity':<15} | {'Type':<10} | {'IDF Score':<10}")
    print("-" * 45)
    
    total_idf = 0.0
    
    # 3. חישוב IDF לכל ישות
    for ent in entities:
        # במקרה של ישות עם כמה מילים (כמו "George Bush"), ניקח את הראשונה או נחשב ממוצע
        first_word = ent.text.split()[0] 
        idf_score = get_word_idf(first_word, idf_array, tokenizer)
        
        print(f"{ent.text:<15} | {ent.label_:<10} | {idf_score:.4f}")
        total_idf += idf_score
        
    avg_idf = total_idf / len(entities)
    print("-" * 45)
    print(f"Average Entity IDF: {avg_idf:.4f}")

# בדיקה על טקסט לדוגמה ש-Qwen יצר
sample_generated_text = "George Bush was the President of the United States in 1992."
analyze_focus_hallucination(sample_generated_text, token_idf, rp_tokenizer, nlp)

Analyzing Text: 'George Bush was the President of the United States in 1992.'

Entity          | Type       | IDF Score 
---------------------------------------------
George Bush     | PERSON     | 4.8472
the United States | GPE        | 1.8811
1992            | DATE       | 5.3729
---------------------------------------------
Average Entity IDF: 4.0337


In [16]:
import torch
import torch.nn.functional as F

def get_qwen_token_entropies(prompt: str, generated_text: str, model, tokenizer):
    # שילוב הפרומפט והתשובה כדי לקבל את ההקשר המלא (בלי זה המודל לא יודע למה הוא עונה)
    full_text = prompt + " " + generated_text
    
    # טוקניזציה והעברה ל-GPU
    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0] # שולפים את הלוגיטים
        
    # חישוב אנטרופיה לכל טוקן: H = -sum(p * log(p))
    probs = F.softmax(logits, dim=-1)
    log_probs = F.log_softmax(logits, dim=-1)
    entropies = -torch.sum(probs * log_probs, dim=-1)
    
    # אנחנו רוצים רק את האנטרופיות של הטקסט שנוצר, אז נחשב כמה טוקנים יש בפרומפט
    prompt_length = len(tokenizer.encode(prompt))
    
    # מבודדים את הטוקנים של התשובה
    gen_tokens = inputs.input_ids[0][prompt_length:]
    
    # מבודדים את האנטרופיות (מזיזים ב-1 כי הלוגיט במקום ה-i מנבא את הטוקן ב-i+1)
    gen_entropies = entropies[prompt_length-1 : -1] 
    
    # חיבור של הטוקנים (כטקסט קריא) יחד עם האנטרופיה שלהם
    token_entropy_mapping = []
    for token_id, ent_val in zip(gen_tokens, gen_entropies):
        word = tokenizer.decode([token_id])
        token_entropy_mapping.append({"word": word, "entropy": ent_val.item()})
        
    return token_entropy_mapping

# בוא נבדוק את זה על הדוגמה שלנו!
test_prompt = "Who is George Bush?"
sample_generated_text = "George Bush was the President of the United States in 1992."

entropies_data = get_qwen_token_entropies(test_prompt, sample_generated_text, base_model, tokenizer)

print(f"{'Token':<12} | {'Entropy (Uncertainty)':<10}")
print("-" * 35)
for item in entropies_data:
    # נדפיס רק מילים עם תוכן, נתעלם מרווחים ריקים
    if item['word'].strip(): 
        print(f"'{item['word']:<10}' | {item['entropy']:.4f}")

Token        | Entropy (Uncertainty)
-----------------------------------
' George   ' | 3.9375
' Bush     ' | 2.4688
' was      ' | 2.0312
' the      ' | 1.5000
' President' | 1.5078
' of       ' | 0.0588
' the      ' | 0.0190
' United   ' | 0.0508
' States   ' | 0.0036
' in       ' | 0.7422
'1         ' | 0.5195
'9         ' | 0.0308
'9         ' | 0.8086
'2         ' | 1.0625
'.         ' | 2.0625


In [ ]:
def calculate_focus_score(entities, entropies_data, idf_array, rp_tokenizer):
    focus_scores = []
    
    print(f"{'Entity':<18} | {'Max Entropy':<12} | {'IDF':<8} | {'FOCUS Score':<12}")
    print("-" * 58)
    
    for ent in entities:
        # 1. התאמת הטוקנים לישות: בגלל ש-Qwen מפרק מילים (כמו 1992), 
        # נאסוף את כל הטוקנים שמרכיבים את הישות הזו.
        ent_entropies = []
        for item in entropies_data:
            clean_token = item['word'].strip()
            if clean_token and clean_token in ent.text:
                ent_entropies.append(item['entropy'])
                
        # אם ישות מורכבת מכמה טוקנים, ניקח את האנטרופיה המקסימלית 
        # (החוליה החלשה ביותר קובעת את חוסר הוודאות)
        agg_entropy = max(ent_entropies) if ent_entropies else 0.0 #Max Pooling
        
        # 2. שליפת ה-IDF דרך הטוקנייזר של RedPajama
        first_word = ent.text.split()[0]
        idf_score = get_word_idf(first_word, idf_array, rp_tokenizer)
        
        # 3. חישוב הציון: אנטרופיה חלקי IDF
        ent_focus = agg_entropy / (idf_score + 1e-8)
        focus_scores.append(ent_focus)
        
        print(f"{ent.text:<18} | {agg_entropy:<12.4f} | {idf_score:<8.4f} | {ent_focus:<12.4f}")
        
    final_focus = sum(focus_scores) / len(focus_scores) if focus_scores else 0.0
    print("-" * 58)
    print(f"Final FOCUS Hallucination Score: {final_focus:.4f}")
    
    return final_focus

# הרצה על הנתונים שכבר יש לנו בזיכרון:
doc = nlp(sample_generated_text)
calculate_focus_score(doc.ents, entropies_data, token_idf, rp_tokenizer)

Entity             | Max Entropy  | IDF      | FOCUS Score 
----------------------------------------------------------
George Bush        | 3.9375       | 4.8472   | 0.8123      
the United States  | 1.5000       | 1.8811   | 0.7974      
1992               | 1.0625       | 5.3729   | 0.1978      
----------------------------------------------------------
Final FOCUS Hallucination Score: 0.6025


np.float64(0.6024965678654558)